# JSON 01 - The 4 functions  (start here)

**Goal of this file:** after it, you never have to guess which `json.` function to call.

---

## What JSON actually is

A JSON file is **just text**. Nothing more. `company.json` is a `.txt` file with
a strict shape. Python cannot use text - so `json` **translates** that text into
normal Python objects you already know: `dict`, `list`, `str`, `int`, `bool`,
`None`.

That is the whole library. Translation, both directions.

## The type table (learn this once)

| in the JSON text | becomes in Python |
|---|---|
| `{ ... }` object | `dict` |
| `[ ... ]` array | `list` |
| `"hello"` | `str` |
| `12` | `int` |
| `1.5` | `float` |
| `true` / `false` | `True` / `False` |
| `null` | `None` |

Three traps for a Python person:
- JSON keys are **always strings**, and **always in double quotes**. `'name'` is invalid JSON.
- It is `true` / `false` / `null` in lowercase - not `True` / `False` / `None`.
- No trailing comma allowed. `{"a": 1,}` is a syntax error.

## The 4 functions

|  | working with a **file** | working with a **string** |
|---|---|---|
| JSON -> Python | `json.load(f)` | `json.loads(s)` |
| Python -> JSON | `json.dump(obj, f)` | `json.dumps(obj)` |

**The whole trick: the `s` means string.** No `s` = file.
`load`/`loads` = read (JSON in, Python out). `dump`/`dumps` = write.

---

Your data file: **`../data/student.json`**

```json
{
  "name": "Amine",
  "age": 21,
  "isEnrolled": true,
  "gpa": 3.75,
  "nickname": null,
  "courses": ["Python", "Algorithms", "Databases"],
  "address": { "city": "Casablanca", "zip": "20000" }
}
```

## Exercise 1 - Load it

Open `../data/student.json` and turn it into a Python object.

Always use `with open(...) as f:` - it closes the file for you even if something
crashes. And always pass `encoding="utf-8"`, otherwise accented letters break on
Windows.

Print `type(data)`. **Expected: `<class 'dict'>`**

In [2]:
import json
with open("../data/student.json") as f:
    data = json.load(f)


print(type(data))          # <class 'dict'>
print(data)

<class 'dict'>
{'name': 'Amine', 'age': 21, 'isEnrolled': True, 'gpa': 3.75, 'nickname': None, 'courses': ['Python', 'Algorithms', 'Databases'], 'address': {'city': 'Casablanca', 'zip': '20000'}}


## Exercise 2 - Read fields

Once loaded it is a **plain dict**. No JSON knowledge needed anymore - it is the
same `data["key"]` you already use.

Print, one per line:
- the name -> `Amine`
- the age -> `21`
- the **second** course -> `Algorithms`
- the city -> `Casablanca`  (careful: `address` is a dict *inside* the dict)
- how many courses -> `3`

In [10]:
print(f"name : {data["name"]} | nickname : {data["nickname"] if data["nickname"] is not None else "inkonnu"} | age : {data["age"]}"+(f" | courses : {data["courses"]}" if data["isEnrolled"] else "") + f" | address : {data['address']}")

name : Amine | nichname : inkonnu | age : 21 | courses : ['Python', 'Algorithms', 'Databases'] | address : {'city': 'Casablanca', 'zip': '20000'}


## Exercise 3 - Prove the type table

For every key, print the key and `type(value).__name__`.

**Expected exactly:**

```
name         -> str
age          -> int
isEnrolled   -> bool
gpa          -> float
nickname     -> NoneType
courses      -> list
address      -> dict
```

Hint: `for key, value in data.items():`

Look at `nickname`. In the file it is `null`. In Python it came back as `None`.
That is the translation happening.

In [13]:
def printJson(data):
    for key , value in data.items():
        print(f"{key} : {value}")
printJson(data)

name : Amine
age : 22
isEnrolled : True
gpa : 3.75
nickname : None
courses : ['Python', 'Algorithms', 'Databases']
address : {'city': 'Casablanca', 'zip': '20000'}


## Exercise 4 - Change it (in memory)

The loaded object is a normal dict, so you edit it normally:

- set `age` to `22`
- add a new key `level` with value `"L3"`
- append `"Networks"` to `courses`

Then print `data`. Nothing is saved to disk yet - you only changed the Python
object. The file is untouched. (Check it: open the file and see.)

In [16]:
#changing and adding some elements to data
data["age"] = 22
data["level"] = "L3"
#priniting data
printJson(data)
#checking the file : there is no change !
with open("../data/student.json","r") as f:
    backup= json.load(f)
printJson(backup)

name : Amine
age : 22
isEnrolled : True
gpa : 3.75
nickname : None
courses : ['Python', 'Algorithms', 'Databases']
address : {'city': 'Casablanca', 'zip': '20000'}
level : L3
name : Amine
age : 21
isEnrolled : True
gpa : 3.75
nickname : None
courses : ['Python', 'Algorithms', 'Databases']
address : {'city': 'Casablanca', 'zip': '20000'}


## Exercise 5 - `dumps` : Python -> JSON **string**

`json.dumps(data)` gives you back a `str`.

1. `s = json.dumps(data)` , then print `type(s)` -> `<class 'str'>`
2. print `s` - notice it is all on one line, and `True` became `true`, `None` became `null`
3. print `json.dumps(data, indent=2)` - the readable version
4. print `json.dumps(data, indent=2, sort_keys=True)` - keys alphabetical

Why does `dumps` matter? Because that is what you send over a network, put in a
log, or use to compare two objects as text.

In [34]:
import json

class Student:
    def __init__(self, name, age, level):
        self.name = name
        self.age = age
        self.level = level
        self.children = []      # list of Student objects

    def addChild(self, student):
        self.children.append(student)

class MyEncoder(json.JSONEncoder):
    def default(self, obj):
        return obj.__dict__

root = Student("Mohamed", 15, "L1")
amine = Student("Amine", 18, "L2")
nadia = Student("Nadia", 21, "M1")
sara = Student("Sara", 20, "L3")
youssef = Student("Youssef", 19, "L2")
omar = Student("Omar", 17, "L1")
lina = Student("Lina", 22, "M2")
root.addChild(amine)
root.addChild(nadia)
amine.addChild(sara)
amine.addChild(youssef)
nadia.addChild(omar)
nadia.addChild(lina)

with open("../data/studentsTree.json", "w") as f:
    json.dump(root, f, cls=MyEncoder, indent=4)

In [33]:
class Student :
    def __init__(self, name, age, level):
        self.name = name
        self.age = age
        self.level = level
class MyEncoder(json.JSONEncoder):
    def default(self, obj):
        return obj.__dict__

StudentList = [Student("Mohamed", 15, "L1"),Student("Amine", 18, "L2"),Student("Sara", 20, "L3"),Student("Youssef", 19, "L2"),Student("Nadia", 21, "M1"), Student("Omar", 17, "L1"),Student("Lina", 22, "M2"),Student("Karim", 18, "L2"),Student("Salma", 20, "L3"),Student("Rachid", 23, "M2")]

with open("../data/studentsTest.json", "w") as f:
    json.dump(StudentList, f, cls=MyEncoder, indent=4)



## Exercise 6 - `dump` : write a real file, then read it back

1. write `data` to `../data/student_out.json` with `indent=2`
   (`with open(..., "w", encoding="utf-8") as f: json.dump(data, f, indent=2)`)
2. open the new file in PyCharm and look at it
3. load it back into `data2`
4. print `data == data2` -> **`True`**

That round-trip (`dump` then `load` gives you back the same thing) is the test
that you understood the whole module.

> **Note the argument order:** `json.dump(obj, f)` - object first, file second.
> Everybody gets this backwards once. `json.load(f)` takes only the file.

In [47]:
with open("../data/student.json") as f:
    data = json.load(f)

data["age"] = 22

with open("../data/student.json", "w") as f:
    json.dump(data,f,indent=4)


## Recap - what to remember from 01

- JSON is text; `json` translates text <-> Python objects.
- `s` = string. `load`/`dump` = file, `loads`/`dumps` = string.
- After `load`, you are just using dicts and lists. That is it.
- `indent=2` for humans, no indent for machines.
- `json.dump(obj, file)` - object first.

Next: **02** - the shape you will meet 90% of the time, a *list of objects*.